# Exercise 1 - Earnings Transcript Sentiment Indicator

This notebook builds the preprocessing pipeline for Exercise 1.

The goal is to construct a 2024 earnings-transcript-based sentiment dataset for firms in the Information Technology and Industrials sectors of the S&P 500.

The notebook covers:
1. Environment setup
2. Loading transcript data
3. Building the sector sample
4. Extracting forward-looking transcript sentences
5. Saving cleaned outputs for sentiment scoring

## 0. Setup imports and project paths

In [2]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

# Resolve project root dynamically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

# Add project root to Python path for local imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: D:\Desk\F550_final_peoject
RAW_DIR: D:\Desk\F550_final_peoject\data\raw
PROCESSED_DIR: D:\Desk\F550_final_peoject\data\processed


## 1. Import local preprocessing functions
Load the preprocessing utilities used to extract speaker-level and sentence-level forward-looking text.

In [3]:
# import self-defined preprocessing functions

from src_ex1.ex1_preprocess import (
    explode_structured_content,
    drop_operator_segments,
    keep_probable_management_segments,
    keep_outlook_segments,
    drop_qna_answer_openers,
    split_segments_into_sentences,
    drop_intro_sentences,
    keep_forward_looking_sentences,
    drop_non_outlook_forward_sentences,
)

## 2. Load transcript data
Load the earnings transcript dataset from Hugging Face parquet storage and restrict the data to calendar year 2024.

In [ ]:
# load full parquet from Hugging Face

parquet_url = "hf://datasets/kurry/sp500_earnings_transcripts/parquet_files/part-0.parquet"

df = pd.read_parquet(parquet_url)
print(df.shape)
df.head()

In [ ]:
# filter 2024 and save raw local copy

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df_2024 = df.loc[
    (df["date"] >= "2024-01-01") &
    (df["date"] < "2025-01-01")
].copy()

df_2024 = df_2024.reset_index(drop=True)

print("2024 transcript rows:", len(df_2024))
print("Min date:", df_2024["date"].min())
print("Max date:", df_2024["date"].max())

In [ ]:
# save raw 2024 transcripts locally

raw_2024_path = RAW_DIR / "transcripts_2024_raw.parquet"
df_2024.to_parquet(raw_2024_path, index=False)

print(f"Saved raw 2024 transcript data to: {raw_2024_path}")

In [5]:
# If it takes too long to extract raw data from hugging face, just run this below
raw_2024_path = RAW_DIR / "transcripts_2024_raw.parquet"
df_2024 = pd.read_parquet(raw_2024_path)

df_2024 = df_2024.reset_index(drop=True)
print(df_2024.shape)
df_2024.head()

(1524, 8)


,symbol,quarter,year,date,content,structured_content,company_name,company_id
0,A,4,2024,2024-11-25 16:30:00,"Operator: Good afternoon. My name is Rob, and I will be your conference operator today. At this time, I would like to welcome everyone to the Agilent Technologies Inc. Fourth Quarter 2024 Earnings...","[{'speaker': 'Operator', 'text': 'Good afternoon. My name is Rob, and I will be your conference operator today. At this time, I would like to welcome everyone to the Agilent Technologies Inc. Four...","Agilent Technologies, Inc.",154924.0
1,A,3,2024,2024-08-21 16:30:00,"Operator: Ladies and gentlemen, welcome to the Agilent Technologies Q3 2024 Earnings Call. My name is Regina and I will be coordinating your call today. [Operator Instructions] I will now hand you...","[{'speaker': 'Operator', 'text': 'Ladies and gentlemen, welcome to the Agilent Technologies Q3 2024 Earnings Call. My name is Regina and I will be coordinating your call today. [Operator Instructi...","Agilent Technologies, Inc.",154924.0
2,A,2,2024,2024-05-29 16:30:00,"Operator: Ladies and gentlemen, welcome to the Agilent Technologies Q2 2024 Earnings Call. My name is Regina, and I will be coordinating your call today. [Operator Instructions]. I will now hand y...","[{'speaker': 'Operator', 'text': 'Ladies and gentlemen, welcome to the Agilent Technologies Q2 2024 Earnings Call. My name is Regina, and I will be coordinating your call today. [Operator Instruct...","Agilent Technologies, Inc.",154924.0
3,A,1,2024,2024-02-27 16:30:00,"Operator: Ladies and gentlemen, welcome to the Agilent Technologies Q1 2024 Earnings Call. My name is Regina and I will be coordinating your call today. [Operator Instructions] I will now hand you...","[{'speaker': 'Operator', 'text': 'Ladies and gentlemen, welcome to the Agilent Technologies Q1 2024 Earnings Call. My name is Regina and I will be coordinating your call today. [Operator Instructi...","Agilent Technologies, Inc.",154924.0
4,AAPL,4,2024,2024-10-31 17:00:00,"Suhasini Chandramouli: Good afternoon, and welcome to the Apple Q4 Fiscal Year 2024 Earnings Conference Call. My name is Suhasini Chandramouli, Director of Investor Relations. Today's call is bein...","[{'speaker': 'Suhasini Chandramouli', 'text': 'Good afternoon, and welcome to the Apple Q4 Fiscal Year 2024 Earnings Conference Call. My name is Suhasini Chandramouli, Director of Investor Relatio...",Apple Inc.,24937.0


## 3. Build the sector sample

### Sample selection note

The transcript universe is first restricted to calendar year 2024.  
Sector labels are then assigned by merging transcript tickers with an S&P 500 constituent table containing GICS sector classifications. The final Exercise 1 sample keeps only firms in the Information Technology and Industrials sectors.

This approach provides a broad sector-level sample for 2024, while acknowledging that the constituent table is not fully point-in-time and may therefore be subject to survivorship bias and index membership change effects.

Merge transcript tickers with an S&P 500 constituent table and keep only Information Technology and Industrials firms.

In [10]:
# load S&P 500 constituent table
sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

sp500 = pd.read_html(
    sp500_url, 
    storage_options={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
)[0]
sp500 = sp500.rename(
    columns={
        "Symbol": "symbol",
        "Security": "security",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "sub_industry",
    }
)

# Standardize ticker formatting
sp500["symbol"] = sp500["symbol"].astype(str).str.replace(".", "-", regex=False)

sp500_small = sp500[["symbol", "security", "sector", "sub_industry"]].copy()
print(sp500_small.shape)
sp500_small.head()

(503, 4)


,symbol,security,sector,sub_industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [11]:
# merge sector information into transcripts

df_2024["symbol"] = df_2024["symbol"].astype(str).str.strip()

df_2024_merged = df_2024.merge(
    sp500_small,
    on="symbol",
    how="left"
)

print("Merged transcript table shape:", df_2024_merged.shape)
print("Rows with missing sector:", df_2024_merged["sector"].isna().sum())

df_2024_merged[["symbol", "company_name", "security", "sector", "sub_industry"]].head(20)

Merged transcript table shape: (1524, 11)
Rows with missing sector: 105


,symbol,company_name,security,sector,sub_industry
0,A,"Agilent Technologies, Inc.",Agilent Technologies,Health Care,Life Sciences Tools & Services
1,A,"Agilent Technologies, Inc.",Agilent Technologies,Health Care,Life Sciences Tools & Services
2,A,"Agilent Technologies, Inc.",Agilent Technologies,Health Care,Life Sciences Tools & Services
3,A,"Agilent Technologies, Inc.",Agilent Technologies,Health Care,Life Sciences Tools & Services
4,AAPL,Apple Inc.,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals"
5,AAPL,Apple Inc.,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals"
6,AAPL,Apple Inc.,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals"
7,AAPL,Apple Inc.,Apple Inc.,Information Technology,"Technology Hardware, Storage & Peripherals"
8,ABBV,AbbVie Inc.,AbbVie,Health Care,Biotechnology
9,ABBV,AbbVie Inc.,AbbVie,Health Care,Biotechnology


In [12]:
# keep target sectors only

target_sectors = ["Information Technology", "Industrials"]

df_ex1_sample = df_2024_merged.loc[
    df_2024_merged["sector"].isin(target_sectors)
].copy()

df_ex1_sample = df_ex1_sample.reset_index(drop=True)

print("Exercise 1 sample shape:", df_ex1_sample.shape)
print("\nTranscript counts by sector:")
print(df_ex1_sample["sector"].value_counts())

print("\nUnique firms by sector:")
print(df_ex1_sample.groupby("sector")["symbol"].nunique().sort_values(ascending=False))

Exercise 1 sample shape: (425, 11)

Transcript counts by sector:
sector
Industrials               228
Information Technology    197
Name: count, dtype: int64

Unique firms by sector:
sector
Industrials               75
Information Technology    66
Name: symbol, dtype: int64


In [13]:
# inspect transcript-level sample

df_ex1_sample[["symbol", "company_name", "sector"]].drop_duplicates().head(30)

,symbol,company_name,sector
0,AAPL,Apple Inc.,Information Technology
4,ACN,Accenture plc,Information Technology
7,ADBE,Adobe Inc.,Information Technology
11,ADI,"Analog Devices, Inc.",Information Technology
15,ADP,"Automatic Data Processing, Inc.",Industrials
18,ADSK,"Autodesk, Inc.",Information Technology
19,AKAM,"Akamai Technologies, Inc.",Information Technology
22,ALLE,Allegion plc,Industrials
25,AMAT,"Applied Materials, Inc.",Information Technology
29,AMD,"Advanced Micro Devices, Inc.",Information Technology


In [14]:
# save transcript-level sample

transcript_sample_path = PROCESSED_DIR / "ex1_transcript_sample_2024.parquet"
df_ex1_sample.to_parquet(transcript_sample_path, index=False)

print(f"Saved transcript-level sample to: {transcript_sample_path}")

Saved transcript-level sample to: D:\Desk\F550_final_peoject\data\processed\ex1_transcript_sample_2024.parquet


## 4. Extract forward-looking transcript text
Apply the preprocessing pipeline to isolate likely forward-looking, outlook-related sentences from earnings calls.

In [15]:
# build speaker-level segments

segments_all = explode_structured_content(df_ex1_sample)
segments_all = drop_operator_segments(segments_all)
segments_all = keep_probable_management_segments(segments_all)
segments_all = keep_outlook_segments(segments_all)
segments_all = drop_qna_answer_openers(segments_all)

print("Filtered speaker-level segments:", len(segments_all))
segments_all.head()

Filtered speaker-level segments: 2896


,symbol,company_name,quarter,year,date,segment_id,speaker,text
13,AAPL,Apple Inc.,4,2024,2024-10-31 17:00:00,13,Tim Cook,"There's a lot there. On Apple Intelligence, we believe it's a compelling upgrade reason. And we'll -- but we just launched it three days ago and so what we've got now from a data point point of vi..."
39,AAPL,Apple Inc.,4,2024,2024-10-31 17:00:00,39,David Vogt,"Right. Okay. So maybe a follow-up for, Tim. When you think about to Luca's point about the rollout being staged over the next several quarters across the world, do you think that has any impact on..."
60,AAPL,Apple Inc.,4,2024,2024-10-31 17:00:00,60,Tim Cook,"Keep in mind that we were have released a lot of APIs and developers will be taking advantage of those APIs. That release has occurred as well and of course more are coming. And so I, definitely b..."
124,AAPL,Apple Inc.,3,2024,2024-08-01 17:00:00,41,Luca Maestri,"And from a gross margin standpoint, as you know, we don't provide any color past the current quarter, and we just provided guidance for the quarter 45.5% to 46.5%. It is essentially broadly in lin..."
132,AAPL,Apple Inc.,3,2024,2024-08-01 17:00:00,49,David Vogt,"Yes. No, that's helpful. I appreciate it. And Luca, just maybe -- I know you didn't give a full rundown of product categories in your prepared remarks. But if I kind of take your comments at face ..."


In [16]:
# split into sentence-level rows

sent_all = split_segments_into_sentences(segments_all)
sent_all = drop_intro_sentences(sent_all)
sent_all = keep_forward_looking_sentences(sent_all)
sent_all = drop_non_outlook_forward_sentences(sent_all)

sent_all = sent_all.reset_index(drop=True)

print("Final forward-looking sentence rows:", len(sent_all))
sent_all[["symbol", "company_name", "date", "speaker", "sentence"]].head(20)

Final forward-looking sentence rows: 4526


,symbol,company_name,date,speaker,sentence
0,AAPL,Apple Inc.,2024-10-31 17:00:00,Tim Cook,"On Apple Intelligence, we believe it's a compelling upgrade reason."
1,AAPL,Apple Inc.,2024-10-31 17:00:00,Tim Cook,"And so, if you look at how we've done this year, we did that very quickly on the 16, on the 16 Pro family, the Pro and the Pro Max, we've been constrained in October, but we believe that soon we'l..."
2,AAPL,Apple Inc.,2024-10-31 17:00:00,David Vogt,"So, should we see something different, let's say, in the December quarter, the March quarter, the June quarter, et cetera, relative to history, given the timing of the rollout and where customers ..."
3,AAPL,Apple Inc.,2024-10-31 17:00:00,Tim Cook,"And what that does to Services, I'll not forecast, but I would say that from an ecosystem point of view, I think it will be great for the user and the user experience."
4,AAPL,Apple Inc.,2024-08-01 17:00:00,Luca Maestri,"And from a gross margin standpoint, as you know, we don't provide any color past the current quarter, and we just provided guidance for the quarter 45.5% to 46.5%."
5,AAPL,Apple Inc.,2024-08-01 17:00:00,David Vogt,"But if I kind of take your comments at face value, I guess what I'm trying to think about is for the next quarter, it sounds like with Services being relatively strong and FX easing a little bit."
6,AAPL,Apple Inc.,2024-08-01 17:00:00,David Vogt,"And so I'm just trying to get a sense for, what are the puts and takes in that sort of outlook particularly as you have Apple Intelligence hopefully stoking the fire for demand going forward?"
7,AAPL,Apple Inc.,2024-08-01 17:00:00,Tim Cook,"Our objective that we said in June is to roll out US English starting in the fall and that is to users, and then proceed with more functionality, more features, if you will, and more languages and..."
8,AAPL,Apple Inc.,2024-05-02 17:00:00,Mike Ng,"And I wanted to ask about, as Apple leans more into AI and Generative AI, should we expect any changes to the historical CapEx cadence that we've seen in the last few years of about $10 billion to..."
9,AAPL,Apple Inc.,2024-05-02 17:00:00,Luca Maestri,It's a model that has worked well for us historically and we plan to continue along the same lines going forward.


In [17]:
# merge sector information back into sentence-level data

sentence_sector_cols = df_ex1_sample[
    ["symbol", "company_name", "date", "sector", "sub_industry"]
].drop_duplicates()

sent_all = sent_all.merge(
    sentence_sector_cols,
    on=["symbol", "company_name", "date"],
    how="left"
)

print(sent_all.shape)
sent_all[["symbol", "company_name", "sector", "date", "speaker", "sentence"]].head(20)

(4526, 11)


,symbol,company_name,sector,date,speaker,sentence
0,AAPL,Apple Inc.,Information Technology,2024-10-31 17:00:00,Tim Cook,"On Apple Intelligence, we believe it's a compelling upgrade reason."
1,AAPL,Apple Inc.,Information Technology,2024-10-31 17:00:00,Tim Cook,"And so, if you look at how we've done this year, we did that very quickly on the 16, on the 16 Pro family, the Pro and the Pro Max, we've been constrained in October, but we believe that soon we'l..."
2,AAPL,Apple Inc.,Information Technology,2024-10-31 17:00:00,David Vogt,"So, should we see something different, let's say, in the December quarter, the March quarter, the June quarter, et cetera, relative to history, given the timing of the rollout and where customers ..."
3,AAPL,Apple Inc.,Information Technology,2024-10-31 17:00:00,Tim Cook,"And what that does to Services, I'll not forecast, but I would say that from an ecosystem point of view, I think it will be great for the user and the user experience."
4,AAPL,Apple Inc.,Information Technology,2024-08-01 17:00:00,Luca Maestri,"And from a gross margin standpoint, as you know, we don't provide any color past the current quarter, and we just provided guidance for the quarter 45.5% to 46.5%."
5,AAPL,Apple Inc.,Information Technology,2024-08-01 17:00:00,David Vogt,"But if I kind of take your comments at face value, I guess what I'm trying to think about is for the next quarter, it sounds like with Services being relatively strong and FX easing a little bit."
6,AAPL,Apple Inc.,Information Technology,2024-08-01 17:00:00,David Vogt,"And so I'm just trying to get a sense for, what are the puts and takes in that sort of outlook particularly as you have Apple Intelligence hopefully stoking the fire for demand going forward?"
7,AAPL,Apple Inc.,Information Technology,2024-08-01 17:00:00,Tim Cook,"Our objective that we said in June is to roll out US English starting in the fall and that is to users, and then proceed with more functionality, more features, if you will, and more languages and..."
8,AAPL,Apple Inc.,Information Technology,2024-05-02 17:00:00,Mike Ng,"And I wanted to ask about, as Apple leans more into AI and Generative AI, should we expect any changes to the historical CapEx cadence that we've seen in the last few years of about $10 billion to..."
9,AAPL,Apple Inc.,Information Technology,2024-05-02 17:00:00,Luca Maestri,It's a model that has worked well for us historically and we plan to continue along the same lines going forward.


In [18]:
# quick summary of cleaned sentence sample

print("Sentence counts by sector:")
print(sent_all["sector"].value_counts())

print("\nUnique firms represented in forward-looking sentence sample:")
print(sent_all.groupby("sector")["symbol"].nunique().sort_values(ascending=False))

Sentence counts by sector:
sector
Industrials               2828
Information Technology    1698
Name: count, dtype: int64

Unique firms represented in forward-looking sentence sample:
sector
Industrials               75
Information Technology    66
Name: symbol, dtype: int64


## 5. Save cleaned Exercise 1 inputs
Export both the transcript-level sector sample and the sentence-level forward-looking dataset for downstream sentiment scoring.

In [19]:
# save sentence-level forward-looking sample

sentence_sample_path = PROCESSED_DIR / "ex1_forward_looking_sentences_2024.parquet"
sent_all.to_parquet(sentence_sample_path, index=False)

print(f"Saved sentence-level forward-looking sample to: {sentence_sample_path}")

Saved sentence-level forward-looking sample to: D:\Desk\F550_final_peoject\data\processed\ex1_forward_looking_sentences_2024.parquet


In [20]:
# final checkpoint

print("Raw 2024 transcript file:")
print(raw_2024_path)

print("\nProcessed transcript-level sample:")
print(transcript_sample_path)

print("\nProcessed sentence-level sample:")
print(sentence_sample_path)

Raw 2024 transcript file:
D:\Desk\F550_final_peoject\data\raw\transcripts_2024_raw.parquet

Processed transcript-level sample:
D:\Desk\F550_final_peoject\data\processed\ex1_transcript_sample_2024.parquet

Processed sentence-level sample:
D:\Desk\F550_final_peoject\data\processed\ex1_forward_looking_sentences_2024.parquet
